# 100 kHz helpers: two matrix weights and one fine-mode TDC

This is a **helper reference**, not the experiment notebook. Running all its
code cells imports packages and defines functions; it does **not** connect to
the board or enable outputs. The call sequence for the future demonstration
notebook is shown as a readable example at the end.

- OUT1 controls matrix weight 1; OUT2 controls matrix weight 2.
- Static DC and hardware-generated time-varying weights are supported.
- The external AWG modulates the matrix input and is not controlled here.
- DIO1_P/E1 pin 5 supplies a 100 kHz, 50% launch to the PCB.
- One PCB pulse return goes to SINGLE input DIO2_P/E1 pin 7, with common ground.

**Fine mode is the intended path.** Upload the qualified V18 RISE/FALL LUTs
once after loading the exact implementation, verify readback, and activate fine
mode. Configure the frame offset once. Subsequent captures reuse both settings.
LUT calibration improves fine precision; it does not remove every systematic
PCB/pulse-width error. The 250 MHz TDC sample clock remains internal.

Install `numpy` and `matplotlib`. See [SCPI_COMMANDS.md](SCPI_COMMANDS.md) for
units, command syntax and record flags. Keep weight voltages within the PCB's
range; these helpers use `INF` load compensation and a conservative 1 V
amplitude-plus-offset limit. Actual pin voltages depend on the physical load.

The PCB return is triggered by the FPGA launch: its launch-to-rise delay is
expected to be fixed and short compared with the 10,000 ns frame. The earlier
AWG characterisation used independent clocks; its drifting edge phase and
phase-retry yield do not describe this topology. The external AWG here only
modulates the matrix input. It is not the TDC timing reference.

Fixed delay is not necessarily zero delay. Keep both return edges away from
frame boundaries over the full input/weight range. A common delay of both edges
cancels in pulse width; it still determines which frame contains the pulse.


In [ ]:
# Settings for the future experiment notebook. Defining them does not contact hardware.
from pathlib import Path
import os
import socket, time, math, json, csv, hashlib
from collections import Counter
from datetime import datetime, timezone
import numpy as np
import matplotlib.pyplot as plt

RP_HOST = "169.254.77.151"
RP_PORT = 5000
BITSTREAM_LABEL = "v18_20261026_140711"
FRAME_PERIOD_NS = 10_000  # 100 kHz; HIGH is 5,000 ns for the WIDTH API.
FRAME_OFFSET_NS = 0      # PCB starting boundary; choose once from PCB return timing, on a 4 ns grid.
FRAMES = 4096            # 40.96 ms; capacity is 16,384.

if os.environ.get("REDPITAYA_DATA_DIR"):
    DATA_ROOT=Path(os.environ["REDPITAYA_DATA_DIR"]).expanduser()
elif os.name=="nt":
    DATA_ROOT=Path.home()/"RedPitaya_local_data"/"RedPitaya_app"
else:
    DATA_ROOT=Path.home()/".local"/"share"/"redpitaya"/"RedPitaya_app"
if not DATA_ROOT.is_absolute():raise ValueError("Use an absolute data directory")
RESULTS_DIR=DATA_ROOT/"100kHz_demo"

# Copy the whole 100kHz_demo folder into another repository. These inputs
# live beside this notebook, independently of the machine-local results.
# Start Jupyter in that folder or its repository root; adjust DEMO_DIR if needed.
DEMO_DIR=next((p for p in (Path.cwd(),Path.cwd()/"100kHz_demo")
               if (p/"v18_luts.json").is_file()),Path.cwd()/"100kHz_demo")
CALIBRATION_FILE=DEMO_DIR/"v18_luts.json"
FRAME_OFFSETS_FILE=DEMO_DIR/"v18_frame_offsets.json"
# Only reuse with the exact V18 route and SINGLE input documented in
# v18_calibration_manifest.json. VERSION18 alone is not a placement identity.


## 1. Connection and stopping outputs

`SCPI.write()` sends one setter and checks the error queue. `SCPI.ask()` reads
one complete reply and checks errors. `stop_outputs()` tries all output stops
and the WIDTH abort, then checks output enables. It returns any cleanup errors.
Closing a TCP connection alone does not turn the FPGA outputs off.


In [ ]:
class SCPI:
    """One sequential TCP session; use `with SCPI(host) as rp:` to close it."""
    def __init__(self, host, port=5000, timeout=5.0):
        self.sock = socket.create_connection((host, port), timeout=timeout)
        self.sock.settimeout(timeout)
        self.buffer = bytearray()

    def _send(self, command):
        if "\n" in command or "\r" in command:
            raise ValueError("Send one SCPI command at a time")
        self.sock.sendall((command+"\r\n").encode("ascii"))

    def _query(self, command):
        self._send(command)
        while b"\n" not in self.buffer:
            packet = self.sock.recv(65536)
            if not packet:
                raise ConnectionError("SCPI connection closed during "+command)
            self.buffer.extend(packet)
            if len(self.buffer)>2_000_000:
                raise RuntimeError("Unexpectedly large SCPI response")
        line, _, remainder = self.buffer.partition(b"\n")
        self.buffer = bytearray(remainder)
        return line.decode("ascii").strip()

    def check(self, context=""):
        reply = self._query("SYST:ERR:NEXT?")
        if int(reply.split(",",1)[0]) != 0:
            raise RuntimeError(f"{context}: SCPI {reply}")

    def write(self, command):
        self._send(command)
        self.check(command)

    def ask(self, command):
        reply = self._query(command)
        self.check(command)
        return reply

    def close(self):
        self.sock.close()

    def __enter__(self): return self
    def __exit__(self, *args): self.close()

def is_on(reply):
    return str(reply).strip().upper() in ("1", "ON")

def wait_width_idle(rp, timeout_s):
    """Return the final status; processing may continue after sampling ends."""
    deadline = time.monotonic()+timeout_s
    while True:
        status = int(rp.ask("TDC:WIDTH:STATUS?"))
        if not status & 1:
            return status
        if time.monotonic()>=deadline:
            raise TimeoutError("WIDTH remained busy; check PCB return and launch")
        time.sleep(0.01)

def stop_outputs(rp):
    """Try every stop even if one fails. Return errors instead of hiding them."""
    errors=[]
    for command in ("PULSE:OUTPUT OFF", "OUTPUT1:STATE OFF", "OUTPUT2:STATE OFF", "TDC:WIDTH:ABORT"):
        try: rp.write(command)
        except Exception as exc: errors.append(f"{command}: {exc}")
    for query in ("PULSE:OUTPUT?", "OUTPUT1:STATE?", "OUTPUT2:STATE?"):
        try:
            if is_on(rp.ask(query)): errors.append(query+" still ON after stop")
        except Exception as exc: errors.append(f"{query}: {exc}")
    return errors

def stop_board(host=RP_HOST, port=RP_PORT):
    """Reconnect and stop all outputs after an interrupted experiment."""
    with SCPI(host,port) as rp:
        errors=stop_outputs(rp)
    if errors:raise RuntimeError("; ".join(errors))
    print("OUT1/OUT2 and launch OFF; WIDTH abort requested.")


## 2. Prepare the TDC and configure launch/frame placement

Call these once during initial setup. `prepare_tdc()` stops outputs, checks
V18 and selects SINGLE while competing legacy test engines are disabled.
`configure_launch()` programs the digital period and 50% duty while OFF.
`set_frame_offset()` programs and reads back the boundary shift.

Neither helper uploads LUTs or starts a capture. Repeat setup after an FPGA
reload. During normal captures leave the offset alone; it is not a width-bias
correction. Changing PCB timing/rate may require a newly checked offset.

Start with `FRAME_OFFSET_NS = 0` for the FPGA-triggered PCB. The bundled
offset map records independent-AWG tests, not a PCB delay calibration; the
demo does not automatically select from it. Once PCB timing is known, optionally
call `check_pcb_return_window()` before capture. Supply bounds covering all
intended weights/inputs and jitter, measured relative to the TDC launch/frame
reference (including path delay), plus the maximum return width. This is a
geometry check, not a measurement or automatic calibration.

For the same-launch frame, require:
`offset + guard < earliest rise` and
`latest rise + maximum width < offset + period - guard`.
If the pulse crosses a boundary, choose one suitable boundary or adjust PCB
timing/width; do not hunt random AWG phases or silently retry captures.


In [ ]:
def prepare_tdc(rp):
    """Stop outputs and select the one V18 SINGLE return path."""
    errors=stop_outputs(rp)
    if errors:raise RuntimeError("; ".join(errors))
    wait_width_idle(rp,2.0)
    image=dict(version=int(rp.ask("TDC:WIDTH:VERSION?")),
               tick_ps=int(rp.ask("TDC:WIDTH:TICK?")),
               taps=int(rp.ask("TDC:TAPS?")),
               capacity=int(rp.ask("TDC:WIDTH:CAPACITY?")),
               minimum_period_ns=int(rp.ask("TDC:WIDTH:MINIMUM:PERIOD?")))
    if (image['version'],image['tick_ps'],image['taps'])!=(18,4000,512):
        raise RuntimeError(f"Expected V18, 4000 ps, 512 taps; got {image}")
    for command in ("TDC:ENABLE OFF","TDC:TEST:INTERNAL OFF",
                    "TDC:MMCM:ENABLE OFF","TDC:INPUT SINGLE"):
        rp.write(command)
    if rp.ask("TDC:INPUT?").upper()!='SINGLE':raise RuntimeError("Input mismatch")
    if rp.ask("PULSE:PIN?").upper()!='DIO1_P':raise RuntimeError("Launch pin mismatch")
    return image

def configure_launch(rp, period_ns=10_000):
    """Program 50% launch timing, in ns. Does not enable the launch."""
    minimum=int(rp.ask("TDC:WIDTH:MINIMUM:PERIOD?"))
    if type(period_ns)!=int or period_ns%4 or not minimum<=period_ns<=8_589_934_588:
        raise ValueError("Period must be on a 4 ns grid within hardware range")
    rp.write("PULSE:OUTPUT OFF")
    rp.write(f"PULSE:TIMING {period_ns},{period_ns//2}")
    if rp.ask("PULSE:TIMING?")!=f"{period_ns},{period_ns//2}":
        raise RuntimeError("Launch timing readback mismatch")

def set_frame_offset(rp, offset_ns=0):
    """Set once while idle; capture_burst never changes this setting."""
    period_ns=int(rp.ask("PULSE:PERIOD?"))
    if type(offset_ns)!=int or offset_ns%4 or not 0<=offset_ns<min(period_ns,262144):
        raise ValueError("Offset needs a 4 ns grid, <=262140 ns, and < period")
    rp.write(f"TDC:WIDTH:OFFSET {offset_ns}")
    if int(rp.ask("TDC:WIDTH:OFFSET?"))!=offset_ns:raise RuntimeError("Offset mismatch")

def read_frame_offset(path, period_ns):
    """Read historical AWG settings, not PCB latency; flag unverified defaults."""
    data=json.loads(Path(path).read_text(encoding='utf-8'))
    if data['bitstream_label']!=BITSTREAM_LABEL or data['input']!='SINGLE':
        raise ValueError("Frame-offset build/input mismatch")
    key=str(period_ns)
    if key in data['offsets_by_period_ns']:
        return int(data['offsets_by_period_ns'][key])
    if key in data['unverified_defaults_by_period_ns']:
        print(f"Offset for {period_ns} ns frames is an UNVERIFIED starting value; check PCB framing.")
        return int(data['unverified_defaults_by_period_ns'][key])
    raise ValueError("No saved offset for this period; measure and set it explicitly")

def check_pcb_return_window(delay_min_ns, delay_max_ns, max_width_ns, *,
                            period_ns=10_000, offset_ns=0, guard_ns=8):
    """Check supplied PCB timing bounds; no hardware access or phase adjustment.

    Delay bounds use the TDC launch/frame reference. The default 8 ns guard
    is a starting engineering margin, not a measured PCB jitter guarantee.
    """
    values=(delay_min_ns,delay_max_ns,max_width_ns,period_ns,offset_ns,guard_ns)
    if not all(math.isfinite(v) for v in values):
        raise ValueError('Timing bounds must be finite')
    if not (0 <= delay_min_ns <= delay_max_ns and max_width_ns > 0
            and period_ns > 0 and 0 <= offset_ns < period_ns and guard_ns >= 0):
        raise ValueError('Invalid PCB timing bounds')
    rise_margin=delay_min_ns-offset_ns
    fall_margin=offset_ns+period_ns-delay_max_ns-max_width_ns
    if min(rise_margin,fall_margin) <= guard_ns:
        raise ValueError('PCB return can reach a frame boundary; check fixed timing/width/offset')
    return dict(rise_margin_ns=rise_margin,fall_margin_ns=fall_margin,guard_ns=guard_ns)


## 3. Load/upload the two LUTs, then enable fine mode

`read_calibration()` only reads the PC file. `upload_luts()` checks the live
version/input and copies both tables, reading every entry back. It leaves fine
mode OFF until `enable_fine_mode()` explicitly turns it ON.

Perform these once after reload (or when intentionally replacing calibration).
Do not upload on every capture. The FPGA retains the tables/activation between
bursts, even if the TCP session closes. A reset/reload or another application
can invalidate that state: verify it before acquisition and restore as needed.
Use only tables from the **exact loaded route/input**; the old saved V18 file
has a placeholder build label, so its provenance must be recorded separately
using the known bitstream label above. No new calibration is generated here.

Portable inputs are included beside this notebook: `v18_luts.json`,
`v18_frame_offsets.json`, and `v18_calibration_manifest.json` (hashes/provenance).
Copy the folder as a unit; no old PC results directory is needed. Saved offsets
are 0 ns at periods 1000, 200, 100, 52 and 40 ns, and 12 ns at 68 ns. These
worked for selected test bursts, not as universal phase-lock settings. For
10,000 ns frames (100 kHz), 0 ns is explicitly an **unverified starting value**.
There is no qualified universal width-bias offset in this bundle.


In [ ]:
def read_calibration(path):
    """Read a qualified candidate_luts.json or uploaded_luts.json on the PC."""
    data=json.loads(Path(path).read_text(encoding='utf-8'))
    calibration=data.get('candidate',data)
    if not calibration.get('ready') or calibration.get('provisional'):
        raise ValueError("Use qualified, nonprovisional RISE/FALL calibration")
    image=calibration['image']
    if (image.get('width_version'),image.get('tick_ps'),image.get('taps'),calibration.get('input'))!=(18,4000,512,'SINGLE'):
        raise ValueError("Calibration must be V18/4000 ps/512 taps/SINGLE")
    for edge in ('RISE','FALL'):
        if calibration['quality'][edge]['errors']:raise ValueError(f"{edge} quality errors")
        values=calibration['tables'][edge]
        if len(values)!=512 or any(type(v)!=int or not 0<=v<=65535 for v in values):
            raise ValueError(f"Invalid {edge} table")
    calibration['_file']=str(path)
    calibration['_sha256']=hashlib.sha256(Path(path).read_bytes()).hexdigest()
    return calibration

def upload_luts(rp, calibration):
    """Once per FPGA reload: upload both tables and verify every entry."""
    if (int(rp.ask("TDC:WIDTH:VERSION?")),int(rp.ask("TDC:WIDTH:TICK?")),
        int(rp.ask("TDC:TAPS?")),rp.ask("TDC:INPUT?").upper())!=(18,4000,512,'SINGLE'):
        raise RuntimeError("Loaded image/input differs from this calibration")
    rp.write("TDC:WIDTH:CALIBRATION:ACTIVATE OFF")
    for edge in ('RISE','FALL'):
        values=calibration['tables'][edge]
        for start in range(0,512,64):
            chunk=values[start:start+64]
            rp.write(f"TDC:WIDTH:CALIBRATION:LUT:WRITE {edge},{start},"+','.join(map(str,chunk)))
            readback=rp.ask(f"TDC:WIDTH:CALIBRATION:LUT:READ? {edge},{start},{len(chunk)}")
            if list(map(int,readback.split(',')))!=chunk:raise RuntimeError("LUT readback mismatch")

def enable_fine_mode(rp):
    """Activate the already uploaded tables; do not upload anything."""
    rp.write("TDC:WIDTH:CALIBRATION:ACTIVATE ON")
    if not is_on(rp.ask("TDC:WIDTH:CALIBRATION:ACTIVATE?")):
        raise RuntimeError("Fine mode did not activate")


## 4. Configure each weight separately

`initialize_weights()` stops/resets the two standard ASGs; it does not reset
the TDC or its LUTs. Call `set_dc_weight()` for a static voltage or
`set_dynamic_weight()` for a hardware waveform. Each setter leaves that channel
OFF and returns its readback. Changing a weight does not change TDC calibration.
`VOLT` is peak amplitude, not Vpp; DC uses zero amplitude and a voltage offset.


In [ ]:
def initialize_weights(rp):
    """Once at setup: reset both ASGs while disabled."""
    rp.write("OUTPUT:STATE OFF")
    rp.write("GEN:RST")

def _prepare_weight(rp, channel):
    if type(channel)!=int or channel not in (1,2):raise ValueError("Choose OUT1 or OUT2")
    # Clear the old amplitude/offset before programming a different level.
    for command in (f"OUTPUT{channel}:STATE OFF",f"SOUR{channel}:VOLT 0",
                    f"SOUR{channel}:VOLT:OFFS 0",f"SOUR{channel}:LOAD INF",
                    f"SOUR{channel}:BURS:STAT CONTINUOUS",f"SOUR{channel}:TRIG:SOUR INT"):
        rp.write(command)

def weight_settings(rp, channel):
    """Read back one channel for display and saved experiment metadata."""
    prefix=f"SOUR{channel}"
    return dict(function=rp.ask(prefix+":FUNC?"),
                amplitude_v=float(rp.ask(prefix+":VOLT?")),
                offset_v=float(rp.ask(prefix+":VOLT:OFFS?")),
                frequency_hz=float(rp.ask(prefix+":FREQ:FIX?")))

def set_dc_weight(rp, channel, voltage_v):
    """Static weight: voltage in V, channel 1 or 2; output remains OFF."""
    if not math.isfinite(voltage_v) or abs(voltage_v)>1:raise ValueError("DC level must be within +/-1 V")
    _prepare_weight(rp,channel)
    rp.write(f"SOUR{channel}:FUNC DC")
    rp.write(f"SOUR{channel}:VOLT:OFFS {voltage_v:.12g}")
    actual=weight_settings(rp,channel)
    if actual['function'].upper()!='DC' or not math.isclose(actual['offset_v'],voltage_v,abs_tol=1e-6) or actual['amplitude_v']!=0:
        raise RuntimeError("DC weight readback mismatch")
    return actual

def set_dynamic_weight(rp, channel, waveform, frequency_hz, amplitude_v, offset_v, phase_deg=0, duty=0.5):
    """Time-varying weight from the FPGA ASG, without Python timing loops."""
    waveform=waveform.upper()
    if waveform not in ('SINE','TRIANGLE','SQUARE','PWM','SAWU','SAWD'):raise ValueError("Unsupported waveform")
    values=(frequency_hz,amplitude_v,offset_v,phase_deg,duty)
    if not all(math.isfinite(v) for v in values):raise ValueError("Settings must be finite")
    if amplitude_v<0 or abs(offset_v)+amplitude_v>1:raise ValueError("Require amplitude+abs(offset) <= 1 V")
    if not 0<frequency_hz<50_000 or not -360<=phase_deg<=360 or not 0<duty<1:
        raise ValueError("Use 0<f<50 kHz, phase -360..360 degrees, duty 0..1")
    _prepare_weight(rp,channel)
    for command in (f"SOUR{channel}:FUNC {waveform}",f"SOUR{channel}:FREQ:FIX {frequency_hz:.12g}",
                    f"SOUR{channel}:PHAS {phase_deg:.12g}",f"SOUR{channel}:DCYC {duty:.12g}",
                    f"SOUR{channel}:VOLT:OFFS {offset_v:.12g}",f"SOUR{channel}:VOLT {amplitude_v:.12g}"):
        rp.write(command)
    actual=weight_settings(rp,channel)
    if actual['function'].upper()!=waveform:raise RuntimeError("Waveform readback mismatch")
    for key,expected in (('frequency_hz',frequency_hz),('amplitude_v',amplitude_v),('offset_v',offset_v)):
        if not math.isclose(actual[key],expected,rel_tol=1e-5,abs_tol=1e-6):raise RuntimeError(f"{key} mismatch")
    return actual

def start_weights(rp):
    """Align and start the two configured ASGs together."""
    for command in ('PHAS:ALIGN','OUTPUT:STATE ON','SOUR:TRIG:INT'):rp.write(command)
    if not all(is_on(rp.ask(f'OUTPUT{ch}:STATE?')) for ch in (1,2)):
        raise RuntimeError("Both weight outputs must be ON")

def start_launch(rp):
    """Start the already configured digital launch BEFORE arming WIDTH."""
    rp.write('PULSE:OUTPUT ON')
    if not is_on(rp.ask('PULSE:OUTPUT?')):raise RuntimeError("Launch did not enable")


## 5. Read and check records

The low-level read helper respects the 128-record SCPI batch limit.
The summary checks frame IDs, flags and signed RISE-minus-FALL LUT arithmetic.
Invalid frames are counted, not discarded to manufacture a successful burst.
These two functions do not configure outputs, calibration or frame offset.


In [ ]:
FIELDS=('cycle_id','flags','coarse_ticks','rise_raw','fall_raw','width_ps')
FLAG_BITS={0:'VALID',1:'NO_RISE',2:'NO_FALL',3:'HIGH_AT_START',4:'EXTRA_EDGES',
    5:'TRIGGER_TIMEOUT',7:'FINE_INVALID',8:'COARSE_ONLY',9:'CALIBRATED',
    10:'BOUNDARY',11:'SATURATED',12:'OUT_OF_RANGE',13:'LATENCY_OVERFLOW',
    14:'CROSS_FRAME',15:'RETURN_LATE'}
STATUS_BITS={0:'BUSY',1:'CAPTURING',2:'PROCESSING',3:'DONE',4:'TRIGGER_TIMEOUT',
    5:'RATE_ERROR',6:'ABORTED',7:'OVERFLOW'}

def read_width_records(rp, count):
    records=[]
    for start in range(0,count,128):
        size=min(128,count-start)
        reply=rp.ask(f"TDC:WIDTH:DATA? {start},{size}")
        batch=[dict(zip(FIELDS,map(int,row.split(',')))) for row in reply.split(';') if row]
        if len(batch)!=size or any(len(row.split(','))!=6 for row in reply.split(';')):
            raise RuntimeError("Malformed or short width response")
        records.extend(batch)
    return records

def summarize_records(records, requested, tick_ps=4000, fine=False, tables=None, reference_ns=None):
    flags=Counter(); integrity=[]; widths=[]
    for expected, row in enumerate(records,1):
        flag=row['flags']
        flags.update(name for bit,name in FLAG_BITS.items() if flag & (1<<bit))
        if row['cycle_id']!=expected: integrity.append(f"cycle id at {expected}")
        if flag & 1:
            mode=flag & (256|512)
            if flag & (2|4|8|16|32|1024|2048|4096|32768): integrity.append(f"valid+error at {expected}")
            if mode!=(512 if fine else 256): integrity.append(f"width mode at {expected}")
            if not 0<=row['rise_raw']<512 or not 0<=row['fall_raw']<512:
                integrity.append(f"raw code at {expected}"); continue
            expected_ps=row['coarse_ticks']*tick_ps
            fine_valid=bool(flag & (1<<16)) and bool(flag & (1<<24))
            if bool(flag & 128)==fine_valid: integrity.append(f"fine-invalid flag at {expected}")
            if expected_ps>0xffffffff: integrity.append(f"coarse range at {expected}")
            if fine:
                if not (flag & (1<<16) and flag & (1<<24)): integrity.append(f"fine status at {expected}")
                if tables is None: raise ValueError("Fine arithmetic requires the uploaded tables")
                expected_ps+=tables['RISE'][row['rise_raw']]-tables['FALL'][row['fall_raw']]
            if row['width_ps']!=expected_ps: integrity.append(f"width arithmetic at {expected}")
            widths.append(row['width_ps']/1000.0)
        elif row['width_ps']!=0: integrity.append(f"invalid nonzero width at {expected}")
    summary=dict(records=len(records),requested=requested,valid=len(widths),
                 flag_counts=dict(flags),integrity_errors=integrity[:20],integrity_error_count=len(integrity))
    if widths:
        x=np.array(widths);blocks=np.array_split(x,min(8,len(x)))
        summary.update(mean_ns=float(x.mean()),std_ns=float(x.std()),min_ns=float(x.min()),
                       max_ns=float(x.max()),block_mean_span_ns=float(np.ptp([b.mean() for b in blocks])))
        if reference_ns is not None: summary['mean_error_ns']=float(x.mean()-reference_ns)
    summary['all_valid']=len(records)==requested and len(widths)==requested and not integrity
    return summary


## 6. Capture one burst using the existing fine-mode setup

`capture_burst()` only checks fine activation, arms, waits, reads and summarizes.
It does not upload LUTs, write the offset, configure weights or start/stop
outputs. Use a visible `try/finally` in the future experiment notebook to stop
all outputs. It returns unsuccessful captures with errors; there is no automatic
retry. Output changes and signal settling belong in the caller's flow.

This demo assumes a return triggered by the FPGA, not a free-running AWG
return. Leave launch running between captures. An incomplete burst calls for
checking PCB pulse presence, width and fixed placement; independent-clock
phase searching is not part of this flow. No fixed PCB delay or AWG-derived
width-bias correction is subtracted from the reported pulse widths.


In [ ]:
def capture_burst(rp, frames, calibration, timeout_s=10.0):
    """Acquire once with existing fine LUTs/offset. Return records + summary."""
    capacity=int(rp.ask('TDC:WIDTH:CAPACITY?'))
    if type(frames)!=int or not 1<=frames<=capacity:raise ValueError('Invalid frame count')
    if not is_on(rp.ask('TDC:WIDTH:CALIBRATION:ACTIVATE?')):
        raise RuntimeError('Fine mode is OFF: upload LUTs and enable fine mode first')
    period_ns=int(rp.ask('PULSE:PERIOD?'))
    if not math.isfinite(timeout_s) or timeout_s<=frames*period_ns*1e-9+0.1:
        raise ValueError('Capture timeout must exceed sampling duration plus 0.1 s')
    result=dict(started_utc=datetime.now(timezone.utc).isoformat(),requested_frames=frames,
                frame_period_ns=period_ns,frame_offset_ns=int(rp.ask('TDC:WIDTH:OFFSET?')),
                timing_topology='fpga_triggered_pcb',automatic_retries=0,
                fine_requested=True,records=[],errors=[],cleanup_errors=[],success=False,
                calibration_file=calibration.get('_file'),calibration_sha256=calibration.get('_sha256'))
    try:
        rp.write(f'TDC:WIDTH:ARM {frames}')
        status=wait_width_idle(rp,timeout_s)
        result['status']=status
        result['status_flags']=[name for bit,name in STATUS_BITS.items() if status&(1<<bit)]
        count=int(rp.ask('TDC:WIDTH:COUNT?'))
        if not 0<=count<=capacity:raise RuntimeError('Invalid record count')
        result['records']=read_width_records(rp,count)
        result['summary']=summarize_records(result['records'],frames,fine=True,tables=calibration['tables'])
        result['success']=bool(status&8) and not bool(status&0xf0) and result['summary']['all_valid']
        if not result['success']:result['errors'].append('Incomplete/invalid burst; inspect status and summary')
    except Exception as exc:
        result['errors'].append(f'{type(exc).__name__}: {exc}')
    result.setdefault('summary',dict(records=len(result['records']),requested=frames,valid=0,all_valid=False))
    result['finished_utc']=datetime.now(timezone.utc).isoformat()
    return result


## 7. Save and plot explicitly

Call `save_capture()` **after cleanup**, attaching any cleanup errors first.
It always saves compact JSON, and saves full raw CSV only for a fully successful
capture. Partial data remains in `result['records']` in memory. Dynamic-weight
signal variation must not be labelled TDC precision.


In [ ]:
def save_capture(result, directory):
    """Full CSV only for fully complete successful captures; compact JSON always."""
    directory=Path(directory);directory.mkdir(parents=True,exist_ok=True)
    stem=directory/datetime.now(timezone.utc).strftime('%Y%m%dT%H%M%S_%fZ')
    metadata={k:v for k,v in result.items() if k!='records'}
    keep=bool(result['success'])
    metadata['raw_records_saved']=keep
    metadata['capture_files']=dict(json=str(stem)+'.json',csv=str(stem)+'.csv' if keep else None)
    if keep:
        with Path(str(stem)+'.csv').open('w',newline='',encoding='utf-8') as f:
            writer=csv.DictWriter(f,fieldnames=FIELDS);writer.writeheader();writer.writerows(result['records'])
    Path(str(stem)+'.json').write_text(json.dumps(metadata,indent=2),encoding='utf-8')
    return metadata['capture_files']

def plot_capture(result):
    """Display every returned record; invalid frames are marked, not averaged in."""
    records=result['records']
    if not records: print('No returned records to plot');return
    valid=np.array([bool(r['flags']&1) for r in records])
    time_ms=(np.array([r['cycle_id'] for r in records])-1)*result['frame_period_ns']/1e6
    widths=np.array([r['width_ps']/1000 if r['flags']&1 else np.nan for r in records])
    fig,axes=plt.subplots(1,2,figsize=(12,3.5),layout='constrained')
    axes[0].plot(time_ms,widths,lw=.7);axes[0].set(xlabel='Nominal frame time (ms)',ylabel='PCB pulse width (ns)')
    if not valid.all(): axes[0].plot(time_ms[~valid],np.zeros((~valid).sum()),'rx',label='Invalid frame');axes[0].legend()
    if valid.any(): axes[1].hist(widths[valid],bins=50)
    axes[1].set(xlabel='Valid pulse width (ns)',ylabel='Count')
    title='Complete burst' if result['success'] else 'INCOMPLETE / ERROR — inspect metadata'
    fig.suptitle(title+'; dynamic-weight variation is not a precision measurement')
    for ax in axes:ax.grid(alpha=.2)
    plt.show()


## Flow for the separate demonstration notebook

The following is documentation, not an executing cell. The future notebook can
reuse these definitions, then make each step visible in its own code cell.

| When | Call | Purpose |
|---|---|---|
| Once, after loading V18 | `prepare_tdc(rp)` | Stop outputs, check image and select SINGLE |
| Once / when changing rate | `configure_launch(rp, 10000)` | Program 100 kHz, 50% launch |
| Once / after choosing a new boundary | `set_frame_offset(rp, FRAME_OFFSET_NS)` | Program/read back the frame offset |
| Once on PC | `calibration = read_calibration(CALIBRATION_FILE)` | Load qualified same-route tables |
| Once after reload | `upload_luts(rp, calibration)` | Upload and read back both edge LUTs |
| Once after upload | `enable_fine_mode(rp)` | Turn fine mode ON |
| At initial weight setup | `initialize_weights(rp)` | Reset both ASGs, leave OFF |
| Before starting / when changing weights | `set_dc_weight(rp, 1, 0.2)` and `set_dc_weight(rp, 2, 0.4)` | Set two independent static weights |
| Optional changing weights | `set_dynamic_weight(rp, 1, 'SINE', 100, 0.1, 0.3)` | Example: 100 Hz, 0.1 V peak, 0.3 V offset |
| Before first capture | `start_weights(rp)` then `start_launch(rp)` | Enable weights and digital launch; allow PCB settling |
| Each capture | `result = capture_burst(rp, FRAMES, calibration)` | Acquire using the existing LUTs/offset |
| Experiment end / any exception | `stop_outputs(rp)` in `finally` | Stop both analogue outputs and launch; abort WIDTH |
| After cleanup | `save_capture(result, RESULTS_DIR)` then `plot_capture(result)` | Save complete raw / compact unsuccessful metadata and inspect |

### One explicit sequence to copy into the future experiment notebook

```python
calibration = read_calibration(CALIBRATION_FILE)  # bundled PC file; same route/input!
offset_ns = FRAME_OFFSET_NS  # PCB boundary, not the historical AWG offset map
# Optional, once measured: check_pcb_return_window(earliest_rise_ns,
#     latest_rise_ns, maximum_width_ns, period_ns=FRAME_PERIOD_NS, offset_ns=offset_ns)
result = None

with SCPI(RP_HOST, RP_PORT) as rp:
    try:
        image = prepare_tdc(rp)
        configure_launch(rp, FRAME_PERIOD_NS)
        set_frame_offset(rp, offset_ns)
        upload_luts(rp, calibration)             # once per FPGA reload
        enable_fine_mode(rp)                     # stays ON between captures

        initialize_weights(rp)
        weight_1 = set_dc_weight(rp, 1, 0.20)
        weight_2 = set_dc_weight(rp, 2, 0.40)
        # Alternative weight 1:
        # weight_1 = set_dynamic_weight(rp, 1, 'SINE', 100, 0.10, 0.30)

        start_weights(rp)
        start_launch(rp)
        time.sleep(0.10)                         # actual PCB settling

        result = capture_burst(rp, FRAMES, calibration)
        result.update(image=image, bitstream_label=BITSTREAM_LABEL,
                      weights={1: weight_1, 2: weight_2})
        print(result['summary'])

        # For another capture with unchanged settings, call only:
        # next_result = capture_burst(rp, FRAMES, calibration)
        # Save each attempt and its own cleanup outcome in the experiment notebook.
    finally:
        cleanup_errors = stop_outputs(rp)
        if result is not None:
            result['cleanup_errors'] = cleanup_errors
            if cleanup_errors:
                result['success'] = False
            result['capture_files'] = save_capture(result, RESULTS_DIR)
        if cleanup_errors:
            print('Cleanup needs attention:', cleanup_errors)

if result is not None:
    plot_capture(result)
```

Stopping PULSE or the ASGs and closing/reopening the TCP session do not require
reuploading calibration. To repeat after stopping, start weights and launch
again, allow settling, and call `capture_burst()`; leave LUTs and offset alone.
For multiple attempts, save each one explicitly; no failed burst should be
hidden by a later good run. After a connection loss, reconnect and call
`stop_board()`. After an FPGA reload, rerun setup/upload/activation and restore
the offset. Fine mode alone does not prove signal identity or PCB accuracy.
